[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tsuxalo/Spoken-Language-Translation-Model/blob/main/capstone_demo.ipynb)

In [ ]:
# Run this cell if executing in Google Colab to install dependencies
import sys
if 'google.colab' in sys.modules:
    !pip install transformers datasets torch torchaudio librosa evaluate jiwer accelerate sentencepiece peft

## 1. Architecture Overview

We didn't design a new model architecture — we fine-tuned OpenAI's pretrained **Whisper (small)**, meaning we took its existing weights (already trained on ~680k hours of multilingual audio) and kept training them specifically on Hausa, rather than starting from nothing.

```
[ RAW AUDIO FILE ]  (e.g., someone speaking Hausa: "Sannu, yaya kake?")
       │
       ▼
========================================================================
                   FEATURE EXTRACTION (signal processing, no learning)
========================================================================
[ Mel-Spectrogram ]
  Converts the sound wave into a visual heat-map of frequencies over time.
  This step has no trainable weights — it's classic audio signal processing,
  done by WhisperProcessor before anything touches the neural network.
       │
       ▼
========================================================================
                     TRANSFORMER ENCODER  ("The Ears + Brain")
========================================================================
[ 2 Conv1D layers -> Multi-Head Self-Attention blocks ]
  The spectrogram first passes through two small convolutional layers
  (this is the actual "CNN" part, and it lives inside the encoder, not
  before it), then through self-attention blocks that scan the whole
  clip for patterns, rhythms, and phonemes. Because Whisper was already
  pretrained on speech in general, it isn't learning "what speech sounds
  like" from scratch — fine-tuning is just adjusting these weights toward
  Hausa-specific sounds.
       │
       ▼
========================================================================
                     TRANSFORMER DECODER  ("The Mouth")
========================================================================
[ Cross-Attention blocks, autoregressive ]
  Looks at the encoder's output and predicts the transcription one token
  (roughly, one sub-word piece) at a time, each prediction conditioned on
  everything predicted so far.
       │
       ▼
[ FINAL OUTPUT TEXT ]  ->  "Sannu, yaya kake?"
```

**Where each script fits in:**
- `data_prep.py` runs the feature-extraction step (audio → mel-spectrogram) and tokenizes the target text, ahead of time, for the whole dataset.
- `train.py` is what actually adjusts the encoder + decoder weights (via `Seq2SeqTrainer`) — this is the fine-tuning step.
- `inference.py` runs a single audio file through the full pipeline above and returns text — this is the same encoder → decoder flow, just at prediction time instead of training time.

## 2. Audio Exploratory Data Analysis

In [ ]:
import io

import numpy as np
import matplotlib.pyplot as plt
import librosa
import librosa.display
import soundfile as sf
from datasets import load_dataset, Audio as HFAudio
from IPython.display import Audio, display

SAMPLING_RATE = 16_000

# Using google/fleurs (ha_ng) instead of mozilla-foundation/common_voice_11_0:
# Mozilla moved Common Voice off the HF Hub to "Mozilla Data Collective" in Oct 2025.
ds = load_dataset("google/fleurs", "ha_ng", split="train")
ds = ds.cast_column("audio", HFAudio(sampling_rate=SAMPLING_RATE, decode=False))

samples = [ds[i] for i in range(3)]

fig, axes = plt.subplots(3, 2, figsize=(12, 9))

for i, sample in enumerate(samples):
    audio_array, sr = sf.read(io.BytesIO(sample["audio"]["bytes"]))

    print(f"Sample {i + 1}: {sample['raw_transcription']}")
    display(Audio(audio_array, rate=sr))

    librosa.display.waveshow(audio_array, sr=sr, ax=axes[i, 0])
    axes[i, 0].set_title(f"Sample {i + 1} — Waveform")

    mel_spec = librosa.feature.melspectrogram(y=audio_array, sr=sr, n_mels=80)
    mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)
    librosa.display.specshow(mel_spec_db, sr=sr, x_axis="time", y_axis="mel", ax=axes[i, 1])
    axes[i, 1].set_title(f"Sample {i + 1} — Mel-Spectrogram")

plt.tight_layout()
plt.show()

**What this is showing:** each row above is one Hausa clip — an audio player you can actually listen to, its **waveform** (left: loudness over time — you can spot pauses between words and where the speaker talks louder or softer), and its **mel-spectrogram** (right: which frequencies/pitches are present at each moment, on a scale roughly matching human hearing). The waveform is intuitive to read but not very informative to a model; the mel-spectrogram is the format that actually gets fed into Whisper, so this is a preview of what the model "sees" instead of what we hear.

## 3. Data Pipeline Verification

## 4. Training Telemetry

In [ ]:
import glob
import json
import os

import matplotlib.pyplot as plt

# If you've trained locally, this cell uses your own trainer_state.json.
# Otherwise it falls back to the actual log from our completed training run
# (3 epochs, whisper-small fine-tuned on FLEURS ha_ng), so this cell tells
# the real story without requiring anyone to retrain first.
OUTPUT_DIR = "./whisper-small-ha"

checkpoints = sorted(glob.glob(f"{OUTPUT_DIR}/checkpoint-*"), key=lambda p: int(p.rsplit("-", 1)[-1]))
state_path = f"{checkpoints[-1]}/trainer_state.json" if checkpoints else None

if state_path and os.path.exists(state_path):
    with open(state_path) as f:
        log_history = json.load(f)["log_history"]
    source_note = f"(from local run: {state_path})"
else:
    log_history = [
        {"step": 25, "loss": 13.716787109375}, {"step": 50, "loss": 8.277050170898438},
        {"step": 75, "loss": 6.0992279052734375}, {"step": 100, "loss": 5.224979858398438},
        {"step": 125, "loss": 4.726186828613281}, {"step": 150, "loss": 4.1446533203125},
        {"step": 175, "loss": 3.800992736816406}, {"step": 200, "loss": 3.615708312988281},
        {"step": 225, "loss": 3.665782470703125}, {"step": 250, "loss": 3.2732379150390627},
        {"step": 275, "loss": 2.9316110229492187}, {"step": 300, "loss": 2.69990234375},
        {"step": 325, "loss": 2.6898858642578123}, {"step": 350, "loss": 2.690340881347656},
        {"step": 375, "loss": 2.518762969970703}, {"step": 400, "loss": 2.497227783203125},
        {"step": 408, "eval_loss": 0.7497789263725281, "eval_wer": 50.49897645854657},
        {"step": 425, "loss": 2.00542236328125}, {"step": 450, "loss": 1.735965576171875},
        {"step": 475, "loss": 1.7062876892089844}, {"step": 500, "loss": 1.677721710205078},
        {"step": 525, "loss": 1.5956822204589844}, {"step": 550, "loss": 1.5235191345214845},
        {"step": 575, "loss": 1.5360792541503907}, {"step": 600, "loss": 1.6421611022949218},
        {"step": 625, "loss": 1.496900634765625}, {"step": 650, "loss": 1.5323004150390624},
        {"step": 675, "loss": 1.545316162109375}, {"step": 700, "loss": 1.5049240112304687},
        {"step": 725, "loss": 1.4029403686523438}, {"step": 750, "loss": 1.6267938232421875},
        {"step": 775, "loss": 1.3107183837890626}, {"step": 800, "loss": 1.3771661376953126},
        {"step": 816, "eval_loss": 0.7022567987442017, "eval_wer": 45.03582395087001},
        {"step": 825, "loss": 1.24801025390625}, {"step": 850, "loss": 0.9759439086914062},
        {"step": 875, "loss": 0.9333879089355469}, {"step": 900, "loss": 0.8799246978759766},
        {"step": 925, "loss": 0.9355815887451172}, {"step": 950, "loss": 0.8932749938964843},
        {"step": 975, "loss": 0.8723690032958984}, {"step": 1000, "loss": 0.8756769561767578},
        {"step": 1025, "loss": 0.9314622497558593}, {"step": 1050, "loss": 1.0698172760009765},
        {"step": 1075, "loss": 0.8003588104248047}, {"step": 1100, "loss": 0.8603568267822266},
        {"step": 1125, "loss": 0.8490235137939454}, {"step": 1150, "loss": 0.8801094055175781},
        {"step": 1175, "loss": 0.8770323944091797}, {"step": 1200, "loss": 0.8575349426269532},
        {"step": 1224, "eval_loss": 0.7122933268547058, "eval_wer": 44.67758444216991},
    ]
    source_note = "(from our completed training run, embedded here — no local training required)"

print(f"Showing training telemetry {source_note}")

train_steps = [e["step"] for e in log_history if "loss" in e]
train_loss = [e["loss"] for e in log_history if "loss" in e]

eval_steps = [e["step"] for e in log_history if "eval_wer" in e]
eval_wer = [e["eval_wer"] for e in log_history if "eval_wer" in e]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(train_steps, train_loss, marker="o")
axes[0].set_title("Training Loss")
axes[0].set_xlabel("Step")
axes[0].set_ylabel("Loss")

axes[1].plot(eval_steps, eval_wer, marker="o", color="darkorange")
axes[1].set_title("Word Error Rate (WER)")
axes[1].set_xlabel("Step")
axes[1].set_ylabel("WER (%)")

plt.tight_layout()
plt.show()

## 5. Interactive Inference

This section runs the full **cascaded** pipeline: our fine-tuned Whisper transcribes Hausa audio to Hausa text, then [NLLB-200](https://huggingface.co/facebook/nllb-200-distilled-600M) (Meta's pretrained translation model) translates that Hausa text to English — two models chained together, no extra training required for the translation step.

In [ ]:
import io

import jiwer
import pandas as pd
import soundfile as sf
import torch
from datasets import load_dataset, Audio as HFAudio
from IPython.display import Audio, HTML
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    WhisperForConditionalGeneration,
    WhisperProcessor,
)

# ASR: loads our fine-tuned model straight from the Hugging Face Hub — no
# local training required. First run downloads ~970MB (cached after that).
ASR_MODEL_DIR = "nahomazmach/whisper-small-ha"
# MT: pretrained Hausa->English translation model, also no training needed.
MT_MODEL_ID = "facebook/nllb-200-distilled-600M"
SAMPLING_RATE = 16_000

device = "cuda" if torch.cuda.is_available() else "cpu"
processor = WhisperProcessor.from_pretrained(ASR_MODEL_DIR)
asr_model = WhisperForConditionalGeneration.from_pretrained(ASR_MODEL_DIR).to(device)

mt_tokenizer = AutoTokenizer.from_pretrained(MT_MODEL_ID)
mt_model = AutoModelForSeq2SeqLM.from_pretrained(MT_MODEL_ID).to(device)
mt_tokenizer.src_lang = "hau_Latn"
forced_bos_token_id = mt_tokenizer.convert_tokens_to_ids("eng_Latn")

test_ds = load_dataset("google/fleurs", "ha_ng", split="test")
test_ds = test_ds.cast_column("audio", HFAudio(sampling_rate=SAMPLING_RATE, decode=False))
samples = [test_ds[i] for i in range(5)]

rows = []
for sample in samples:
    audio_array, sr = sf.read(io.BytesIO(sample["audio"]["bytes"]))
    input_features = processor.feature_extractor(
        audio_array, sampling_rate=sr, return_tensors="pt"
    ).input_features.to(device)

    predicted_ids = asr_model.generate(input_features, language="hausa", task="transcribe")
    prediction = processor.tokenizer.batch_decode(predicted_ids, skip_special_tokens=True)[0].strip()

    mt_inputs = mt_tokenizer(prediction, return_tensors="pt").to(device)
    translated_ids = mt_model.generate(**mt_inputs, forced_bos_token_id=forced_bos_token_id, max_length=256)
    english = mt_tokenizer.batch_decode(translated_ids, skip_special_tokens=True)[0].strip()

    ground_truth = sample["raw_transcription"]
    wer = jiwer.wer(ground_truth, prediction) * 100
    # Embed a playable audio widget directly as an HTML <audio> tag, so you
    # can listen to each clip right next to its ground truth and prediction.
    audio_html = Audio(audio_array, rate=sr)._repr_html_()

    rows.append({
        "Sample ID": sample["id"],
        "Audio": audio_html,
        "Ground Truth Hausa": ground_truth,
        "Model Output (Hausa)": prediction,
        "English Translation": english,
        "WER Score": round(wer, 2),
    })

results_df = pd.DataFrame(rows)
HTML(results_df.to_html(escape=False))

## 6. Direct Speech Translation (Pilot)

Everything above is a **cascade**: Hausa audio → Hausa text → English text, two models chained together. This section runs something architecturally different — a **direct** model that goes straight from Hausa audio to English text in a single pass, with no Hausa-text step in between at all.

```
[ Hausa audio ]  →  (Whisper encoder/decoder, task="translate", LoRA-adapted)  →  [ English text ]
```

Under the hood it's still `openai/whisper-small` — Whisper natively supports a `task="translate"` mode that outputs English directly, since it was originally trained on translation pairs alongside transcription. We LoRA-fine-tuned that mode on real Hausa-audio-to-English-text pairs (not more Hausa transcriptions) so it gets better specifically at Hausa. LoRA means only a small add-on set of weights (~1.8M parameters, 0.7% of the model) gets trained — the base model stays frozen.

This was a genuine **pilot**, not a full training run: 256 training examples, 50 steps, trained on a targeted subset of the [NaijaS2ST](https://huggingface.co/datasets/McGill-NLP/NaijaS2ST) dataset to avoid an unnecessary ~69GB full-dataset download. Full methodology, the bugs hit and fixed along the way, and the complete results are documented in [`direct_pilot/RESULTS.md`](direct_pilot/RESULTS.md).

In [ ]:
import io

import soundfile as sf
import torch
from datasets import load_dataset, Audio as HFAudio
from peft import PeftModel
from transformers import WhisperForConditionalGeneration, WhisperProcessor

# Loads the pilot LoRA adapter from the Hugging Face Hub, over the base
# (not our Hausa-fine-tuned) whisper-small — no local training required.
DIRECT_ADAPTER_REPO = "nahomazmach/whisper-small-ha-en-direct-pilot"
BASE_MODEL_ID = "openai/whisper-small"
SAMPLING_RATE = 16_000

device = "cuda" if torch.cuda.is_available() else "cpu"
direct_processor = WhisperProcessor.from_pretrained(DIRECT_ADAPTER_REPO)
direct_base_model = WhisperForConditionalGeneration.from_pretrained(BASE_MODEL_ID)
direct_model = PeftModel.from_pretrained(direct_base_model, DIRECT_ADAPTER_REPO).to(device)
direct_model.eval()

# Same FLEURS test clip used in Section 5, so the cascade and direct outputs
# below are directly comparable side by side.
test_ds = load_dataset("google/fleurs", "ha_ng", split="test")
test_ds = test_ds.cast_column("audio", HFAudio(sampling_rate=SAMPLING_RATE, decode=False))
sample = test_ds[0]
audio_array, sr = sf.read(io.BytesIO(sample["audio"]["bytes"]))

input_features = direct_processor.feature_extractor(
    audio_array, sampling_rate=sr, return_tensors="pt"
).input_features.to(device)

predicted_ids = direct_model.generate(
    input_features, language="hausa", task="translate", max_new_tokens=225, num_beams=1
)
direct_english = direct_processor.tokenizer.batch_decode(predicted_ids, skip_special_tokens=True)[0].strip()

print("Ground truth Hausa:  ", sample["raw_transcription"])
print()
print("Direct pilot English:", direct_english)
print()
print("(Compare against the cascade's English output for the same sample ID in Section 5's table above.)")

## 7. Direct vs. Cascade: Results Comparison

Three systems, scored on the same kind of task — translating Hausa audio to English text — using **BLEU** and **chrF++** (standard machine-translation metrics; higher is better for both). These are the actual measured numbers from our runs, embedded here so this comparison doesn't require re-running any evaluation.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

systems = ["Cascade\n(gold Hausa text)", "Cascade\n(real ASR output)", "Direct pilot\n(this run)"]
bleu_scores = [23.5, 9.0, 0.24]
chrf_scores = [48.0, 34.0, 14.39]

x = np.arange(len(systems))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
bars1 = ax.bar(x - width / 2, bleu_scores, width, label="BLEU")
bars2 = ax.bar(x + width / 2, chrf_scores, width, label="chrF++")

ax.set_ylabel("Score")
ax.set_title("Hausa → English Translation Quality: Cascade vs. Direct")
ax.set_xticks(x)
ax.set_xticklabels(systems)
ax.legend()
ax.bar_label(bars1, fmt="%.1f", padding=3)
ax.bar_label(bars2, fmt="%.1f", padding=3)

plt.tight_layout()
plt.show()

**This is a real, legitimate finding:** at small scale, the direct approach underperforms the cascade substantially, suggesting the cascade's advantage from independent massive pretraining (Whisper's 680k hours, NLLB's large parallel-text corpus) outweighs its ASR-error-propagation weakness — at least until a direct model gets enough paired data to compete. This isn't a verdict that direct approaches are worse in general — it's evidence that, in a genuinely low-resource setting like this one, the data-efficiency advantage of a cascade built from two separately pretrained giants currently matters more than avoiding error propagation.

Two caveats worth keeping in mind: the direct pilot's score is measured on a validation split derived from NaijaS2ST's training data, not the exact same held-out `dev` examples the cascade's BLEU 8–10 was measured on — a fully rigorous side-by-side still needs that. And 256 training examples is a tiny fraction of what a direct model would realistically need to be competitive, so this is a feasibility signal, not a final verdict. See [`direct_pilot/RESULTS.md`](direct_pilot/RESULTS.md) for the full write-up.